# Adaptive Portfolio Positioning under Market Volatility

**Weekly Single-Inversion Baseline (Ablation)**  

This notebook implements a parsimonious weekly-frequency ablation that uses only the lagged VIX term-structure inversion ratio as the transition driver.

It serves as a simpler benchmark against the main daily two-factor specification. The goal is to assess how much of the risk-reduction performance can be obtained from a coarser, single-feature weekly rule.


## 1. Environment

In [ ]:
# Environment Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import norm
from scipy.optimize import minimize
from scipy.special import logsumexp
from numba import njit
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.figsize": (14, 6),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("Environment ready. Random seed =", RANDOM_SEED)


## 2. Data (Weekly Aggregation)

In [ ]:
# Data Download + Weekly Aggregation (strict no look-ahead)
import yfinance as yf

def download_close(ticker, start="2010-01-04", end="2026-07-31"):
    """Robust close extractor (handles MultiIndex from yfinance)."""
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        s = raw["Close"]
        if isinstance(s, pd.DataFrame):
            s = s.iloc[:, 0]
        return s
    return raw["Close"]

spy_d   = download_close("SPY")
vix_d   = download_close("^VIX")
vix3m_d = download_close("^VIX3M")
bil_d   = download_close("BIL")

daily = pd.concat([spy_d, vix_d, vix3m_d, bil_d], axis=1, join="inner")
daily.columns = ["SPY", "VIX", "VIX3M", "BIL"]
daily = daily.dropna()
daily["SPY_return"] = daily["SPY"].pct_change()
daily["BIL_return"] = daily["BIL"].pct_change()
daily["Inversion"]  = daily["VIX"] / daily["VIX3M"]
daily = daily.dropna()

# Friday-ended weeks
daily["week"] = daily.index.to_period("W-FRI")
weekly = daily.groupby("week").agg(
    SPY=("SPY", "last"),
    VIX=("VIX", "last"),
    VIX3M=("VIX3M", "last"),
    BIL=("BIL", "last"),
    SPY_return=("SPY_return", lambda x: np.prod(1.0 + x) - 1.0),
    BIL_return=("BIL_return", lambda x: np.prod(1.0 + x) - 1.0),
    Inversion=("Inversion", "last"),
)
# Inversion lagged ONE week for transitions (no look-ahead)
weekly["Inversion_lagged"] = weekly["Inversion"].shift(1)
weekly = weekly.dropna()
weekly.index = weekly.index.to_timestamp(how="end").normalize()

weekly.to_csv("data_weekly.csv")
print(f"Weekly T = {len(weekly)}  |  {weekly.index.min().date()} → {weekly.index.max().date()}")
print(weekly[["SPY_return", "Inversion_lagged", "BIL_return"]].describe().round(4))


## 3. Model Specification (Single-Inversion TVTP)

In [ ]:
# Parsimonious TVTP-MS-GJR-GARCH (K=2)
#          + Asymmetric soft persistence penalty
# Goal: multi-week durations AND economically rare Stress mass.
# Unconstrained or symmetric stay targets often push P(Stress) near 50%.
# Asymmetric targets: Normal highly persistent; Stress persistent when entered
# but less sticky, so unconditional Stress frequency falls toward 15-30%.

returns          = weekly["SPY_return"].values.astype(np.float64)
inversion_lagged = weekly["Inversion_lagged"].values.astype(np.float64)
T = len(returns)
K = 2
print(f"Sample size T = {T}  |  K = {K}")

def unpack_parameters(theta):
    theta = np.asarray(theta, dtype=np.float64)
    mu    = theta[0:K].copy()
    omega = np.exp(theta[K:2*K])
    alpha = np.exp(theta[2*K:3*K])
    gamma = theta[3*K:4*K].copy()
    beta  = 1.0 / (1.0 + np.exp(-theta[4*K:5*K]))

    # Dual identification: descending mean
    order = np.argsort(-mu)
    mu, omega, alpha, gamma, beta = (mu[order], omega[order], alpha[order],
                                     gamma[order], beta[order])
    inter = theta[5*K:5*K + K*K].reshape(K, K)[order, :][:, order]
    s_enter = np.exp(theta[5*K + K*K])   # >= 0
    s_exit  = theta[5*K + K*K + 1]
    slope_vec = np.array([s_enter, s_exit], dtype=np.float64)
    return mu, omega, alpha, gamma, beta, inter, slope_vec


def transition_matrix(inversion, trans_intercept, slope_vec):
    s_enter, s_exit = float(slope_vec[0]), float(slope_vec[1])
    P = np.zeros((K, K))
    logits0 = trans_intercept[0].copy()
    logits0[1] = logits0[1] + s_enter * inversion
    P[0, :] = np.exp(logits0 - logsumexp(logits0))
    logits1 = trans_intercept[1].copy()
    logits1[0] = logits1[0] + s_exit * inversion
    P[1, :] = np.exp(logits1 - logsumexp(logits1))
    return P


def build_P_all(inv, intercept, slope_vec):
    s_enter, s_exit = float(slope_vec[0]), float(slope_vec[1])
    T_ = len(inv)
    logits = np.broadcast_to(intercept, (T_, K, K)).copy()
    logits[:, 0, 1] = logits[:, 0, 1] + s_enter * inv
    logits[:, 1, 0] = logits[:, 1, 0] + s_exit * inv
    m = logits.max(axis=2, keepdims=True)
    e = np.exp(logits - m)
    return e / e.sum(axis=2, keepdims=True)


@njit(cache=False)
def nll_numba(ret, mu, omega, alpha, gamma, beta, P_all):
    T_ = len(ret)
    xi = np.ones(K) / float(K)
    s2 = np.full(K, np.var(ret[:min(50, T_)]))
    ll = 0.0
    for t in range(T_):
        xi_pred = np.zeros(K)
        for j in range(K):
            acc = 0.0
            for i in range(K):
                acc += P_all[t, i, j] * xi[i]
            xi_pred[j] = acc

        dens = np.empty(K)
        s2n  = np.empty(K)
        for k in range(K):
            if t > 0:
                eps  = ret[t - 1] - mu[k]
                Ineg = 1.0 if eps < 0.0 else 0.0
            else:
                eps, Ineg = 0.0, 0.0
            s2n[k] = omega[k] + (alpha[k] + gamma[k] * Ineg) * (eps * eps) + beta[k] * s2[k]
            if s2n[k] < 1e-12:
                s2n[k] = 1e-12
            r = ret[t] - mu[k]
            dens[k] = np.exp(-0.5 * np.log(2.0 * np.pi * s2n[k]) - 0.5 * (r * r) / s2n[k])
            if dens[k] < 1e-300:
                dens[k] = 1e-300

        joint_sum = 0.0
        for k in range(K):
            joint_sum += xi_pred[k] * dens[k]
        if joint_sum < 1e-300:
            return 1e12
        for k in range(K):
            xi[k] = xi_pred[k] * dens[k] / joint_sum
            s2[k] = s2n[k]
        ll += np.log(joint_sum)
    return -ll


def stationary_dist(P):
    """Stationary distribution of a 2x2 row-stochastic matrix."""
    # Solve pi P = pi, sum(pi)=1
    # For 2-state: pi0 = p10 / (p01 + p10), pi1 = p01 / (p01 + p10)
    p01 = float(P[0, 1])
    p10 = float(P[1, 0])
    denom = p01 + p10
    if denom < 1e-12:
        return np.array([0.5, 0.5])
    pi0 = p10 / denom
    return np.array([pi0, 1.0 - pi0])


def negative_log_likelihood(theta):
    """
    NLL + asymmetric soft persistence + soft Stress-mass penalty.

    TARGET_STAY0 high  -> Normal lasts many weeks
    TARGET_STAY1 moderate -> Stress still persistent when entered, but exits faster
    TARGET_STRESS_MASS -> unconditional Stress probability not near 50%
    """
    mu, omega, alpha, gamma, beta, t_int, slope_vec = unpack_parameters(theta)
    if not (omega[0] < omega[1]):
        return 1e12 + 1e6 * max(0.0, omega[0] - omega[1])

    P_all = build_P_all(inversion_lagged, t_int, slope_vec)
    nll = float(nll_numba(returns, mu, omega, alpha, gamma, beta, P_all))

    # Average transition matrix over the sample
    P_avg = P_all.mean(axis=0)
    stay0 = float(P_avg[0, 0])
    stay1 = float(P_avg[1, 1])
    pi_stat = stationary_dist(P_avg)
    stress_mass = float(pi_stat[1])

    # Asymmetric duration targets
    TARGET_STAY0 = 0.94      # Normal ~ 16.7 weeks
    TARGET_STAY1 = 0.82      # Stress ~ 5.6 weeks
    # Unconditional Stress mass target (equity "elevated vol" should be infrequent)
    TARGET_STRESS_MASS = 0.25
    STRESS_MASS_BAND   = 0.05   # soft band around target before penalty grows

    PEN_STAY   = 200.0
    PEN_MASS   = 150.0

    pen_stay = PEN_STAY * (
        max(0.0, TARGET_STAY0 - stay0) ** 2 +
        max(0.0, TARGET_STAY1 - stay1) ** 2
    )
    # Penalize Stress mass above (target + band); mild pull if far below is optional
    excess = max(0.0, stress_mass - (TARGET_STRESS_MASS + STRESS_MASS_BAND))
    shortfall = max(0.0, (TARGET_STRESS_MASS - STRESS_MASS_BAND) - stress_mass)
    pen_mass = PEN_MASS * (excess ** 2 + 0.25 * shortfall ** 2)

    return nll + pen_stay + pen_mass


# JIT warm-up
_ = negative_log_likelihood(np.random.randn(5 * K + K * K + 2) * 0.1)
print("Cell 3 ready: asymmetric persistence + Stress-mass penalty")
print("  TARGET_STAY0=0.94 | TARGET_STAY1=0.82 | TARGET_STRESS_MASS=0.25")


## 4. Parameter Estimation

In [ ]:
# Multi-start MLE (always re-estimate; no pre-loaded theta)
import time

N_STARTS = 16
MAX_ITER = 300
FTOL     = 1e-9

best_nll   = np.inf
best_theta = None
all_nll    = []

print(f"Multi-start MLE (asymmetric persistence): {N_STARTS} starts")
print("=" * 72)
t0 = time.time()

for s in range(N_STARTS):
    np.random.seed(RANDOM_SEED + 300 + s)
    t_start = time.time()

    mu_init    = np.array([0.0025, -0.0030]) + np.random.normal(0.0, 0.0010, K)
    omega_init = np.log(np.array([1.5e-5, 8e-5]) * (0.5 + np.random.rand(K)))
    alpha_init = np.log(np.clip(np.array([0.05, 0.10]) * (0.5 + np.random.rand(K)), 1e-4, 0.45))
    gamma_init = np.array([0.10, 0.25]) + np.random.normal(0.0, 0.05, K)
    beta_raw   = np.clip(np.array([0.80, 0.70]) + np.random.normal(0.0, 0.05, K), 0.50, 0.96)
    beta_init  = np.log(beta_raw / (1.0 - beta_raw))

    # Stronger Normal diagonal in the *starting value* only
    inter_init = np.random.normal(0.0, 0.25, (K, K))
    inter_init[0, 0] = 2.8 + 0.6 * np.random.rand()   # Normal sticky
    inter_init[1, 1] = 1.6 + 0.5 * np.random.rand()   # Stress less sticky

    raw_enter_init = np.random.normal(-0.3, 0.4)
    raw_exit_init  = np.random.normal(0.2, 0.4)

    theta0 = np.concatenate([
        mu_init, omega_init, alpha_init, gamma_init, beta_init,
        inter_init.ravel(),
        np.array([raw_enter_init, raw_exit_init]),
    ])

    try:
        res = minimize(
            negative_log_likelihood, theta0, method="L-BFGS-B",
            options={"maxiter": MAX_ITER, "ftol": FTOL, "disp": False},
        )
        nll = float(res.fun)
        all_nll.append(nll)
        dt = time.time() - t_start
        if np.isfinite(nll) and nll < best_nll:
            best_nll   = nll
            best_theta = res.x.copy()
            print(f"Start {s+1:02d}: NLL+pen = {nll:12.4f}  ({dt:.1f}s)  <- new best")
        else:
            print(f"Start {s+1:02d}: NLL+pen = {nll:12.4f}  ({dt:.1f}s)")
    except Exception as e:
        print(f"Start {s+1:02d}: failed - {str(e)[:80]}")
        all_nll.append(np.nan)

print("=" * 72)
print(f"Done in {(time.time()-t0)/60:.1f} min  |  Best NLL+pen = {best_nll:.6f}")
print(f"Finite starts: {np.sum(np.isfinite(all_nll))}/{N_STARTS}")

if best_theta is None:
    raise RuntimeError("All MLE starts failed.")

np.savez(
    "mle_best_theta_weekly_K2_asymm.npz",
    best_theta=best_theta,
    best_nll=best_nll,
    n_starts=N_STARTS,
    timestamp=np.datetime64("now"),
)
print("Saved -> mle_best_theta_weekly_K2_asymm.npz (optional cache only)")


## 5. Filtering and Diagnostics

In [ ]:
# Unpack, filter, smoother, duration + frequency diagnostics
mu, omega, alpha, gamma, beta, trans_intercept, slope_vec = unpack_parameters(best_theta)

print("=" * 72)
print("Estimated parameters (asymmetric persistence)")
print("=" * 72)
names = ["0 Normal Growth", "1 High-Vol/Stress"]
print(f"{'Regime':<22} {'mu':>10} {'omega':>12} {'alpha':>8} {'gamma':>8} {'beta':>8}")
print("-" * 72)
for k in range(K):
    print(f"{names[k]:<22} {mu[k]:10.6f} {omega[k]:12.2e} {alpha[k]:8.4f} "
          f"{gamma[k]:8.4f} {beta[k]:8.4f}")
print(f"\nslope_enter (>=0) = {slope_vec[0]:.4f}")
print(f"slope_exit        = {slope_vec[1]:.4f}")
print(f"ID: mu_desc={mu[0]>mu[1]}  omega_asc={omega[0]<omega[1]}")

# Hamilton filter
xi_filtered = np.zeros((T, K))
sigma2_path = np.zeros((T, K))
xi_prev = np.ones(K) / K
sigma2_prev = np.full(K, np.var(returns[:min(50, T)]))

for t in range(T):
    P = transition_matrix(inversion_lagged[t], trans_intercept, slope_vec)
    xi_pred = P.T @ xi_prev
    if t > 0:
        eps_prev = returns[t-1] - mu
        I_neg = (eps_prev < 0).astype(np.float64)
    else:
        eps_prev = np.zeros(K)
        I_neg = np.zeros(K)
    sigma2 = omega + (alpha + gamma * I_neg) * (eps_prev ** 2) + beta * sigma2_prev
    sigma2 = np.maximum(sigma2, 1e-12)
    sigma2_path[t] = sigma2
    resid = returns[t] - mu
    log_dens = -0.5 * np.log(2.0 * np.pi * sigma2) - 0.5 * (resid ** 2) / sigma2
    max_ld = np.max(log_dens)
    joint = xi_pred * np.exp(log_dens - max_ld)
    lik_t = joint.sum()
    xi_prev = joint / lik_t if lik_t >= 1e-300 else np.ones(K) / K
    sigma2_prev = sigma2
    xi_filtered[t] = xi_prev

# Kim smoother
xi_smoothed = np.zeros((T, K))
xi_smoothed[-1] = xi_filtered[-1].copy()
for t in range(T - 2, -1, -1):
    P = transition_matrix(inversion_lagged[t + 1], trans_intercept, slope_vec)
    xi_pred = np.maximum(P.T @ xi_filtered[t], 1e-12)
    xi_smoothed[t] = xi_filtered[t] * (P @ (xi_smoothed[t + 1] / xi_pred))
    xi_smoothed[t] /= xi_smoothed[t].sum()

# Average P, duration, stationary mass
P_avg = np.zeros((K, K))
for t in range(T):
    P_avg += transition_matrix(inversion_lagged[t], trans_intercept, slope_vec)
P_avg /= T
exp_dur = 1.0 / np.maximum(1.0 - np.diag(P_avg), 1e-8)
pi_stat = stationary_dist(P_avg)

print("\nAverage transition matrix:")
print(np.round(P_avg, 4))
print("Expected duration (weeks):")
for k in range(K):
    print(f"  {names[k]}: {exp_dur[k]:.2f}  (~{exp_dur[k]*5:.1f} days)")
print(f"Stationary mass: Normal={pi_stat[0]:.3f}  Stress={pi_stat[1]:.3f}")
print(f"Mean filtered mass: Normal={xi_filtered[:,0].mean():.3f}  "
      f"Stress={xi_filtered[:,1].mean():.3f}")

print("\nConditional P by Inversion level:")
for inv_level, label in [(0.80, "low"), (0.90, "mid"), (1.05, "high"), (1.20, "extreme")]:
    P = transition_matrix(inv_level, trans_intercept, slope_vec)
    print(f"  inv={inv_level:.2f} ({label}): stay0={P[0,0]:.3f} stay1={P[1,1]:.3f} "
          f"enter={P[0,1]:.3f} exit={P[1,0]:.3f}")

# Hard classification table
most_likely = np.argmax(xi_smoothed, axis=1)
rows = []
for k in range(K):
    mask = most_likely == k
    r = returns[mask]
    inv = inversion_lagged[mask]
    rows.append({
        "Regime": names[k],
        "N_obs": int(mask.sum()),
        "Frequency": float(mask.mean()),
        "Mean_return": float(r.mean()) if mask.any() else np.nan,
        "Std_return": float(r.std()) if mask.any() else np.nan,
        "CVaR_5pct": float(np.percentile(r, 5)) if mask.any() else np.nan,
        "Mean_Inversion": float(inv.mean()) if mask.any() else np.nan,
    })
char_df = pd.DataFrame(rows).set_index("Regime")
print("\nRegime characteristics (most-likely smoothed):")
print(char_df.round(4))
char_df.to_csv("regime_characteristics_weekly_K2_asymm.csv")

# Success checklist (print pass/fail style)
freq_stress = float(char_df.loc[names[1], "Frequency"])
print("\n--- Refinement checklist ---")
print(f"[ ] Stress frequency in ~15-30%?   now {freq_stress:.1%}")
print(f"[ ] Normal duration > Stress?      {exp_dur[0]:.1f} vs {exp_dur[1]:.1f} weeks")
print(f"[ ] Stress Mean_Inversion higher?  "
      f"{char_df.loc[names[1],'Mean_Inversion']:.3f} vs "
      f"{char_df.loc[names[0],'Mean_Inversion']:.3f}")
print(f"[ ] slope_enter >= 0?              {slope_vec[0]:.4f}")


## 6. Position Mapping and Execution

In [ ]:
# Smoothed regime probabilities + SPY with regime background
# Green = Normal Growth (0), Red/Salmon = High-Vol / Stress (1)
# Background on BOTH panels uses most-likely smoothed state (visual only).

most_likely = np.argmax(xi_smoothed, axis=1)
dates = weekly.index
spy_px = weekly["SPY"].values.astype(float)

fig, axes = plt.subplots(
    2, 1, figsize=(14, 8), sharex=True,
    gridspec_kw={"height_ratios": [1.15, 1.0]},
)

# ----- Panel 1: smoothed probabilities -----
ax = axes[0]
ax.fill_between(
    dates, 0.0, xi_smoothed[:, 0],
    color="green", alpha=0.55, linewidth=0, label="Normal Growth (0)",
)
ax.fill_between(
    dates, xi_smoothed[:, 0], 1.0,
    color="salmon", alpha=0.55, linewidth=0, label="High-Vol / Stress (1)",
)
ax.plot(dates, xi_smoothed[:, 0], color="darkgreen", lw=0.6, alpha=0.8)
ax.set_ylim(0.0, 1.02)
ax.set_ylabel("Smoothed P(regime)")
ax.set_title(
    "Smoothed regime probabilities (asymmetric persistence + Stress-mass penalty)"
)
ax.legend(loc="upper right", framealpha=0.9)

# ----- Panel 2: SPY price with regime background -----
ax = axes[1]

# Shade contiguous regime segments behind the price path
t0 = 0
for t in range(1, T + 1):
    if t == T or most_likely[t] != most_likely[t0]:
        color = "green" if most_likely[t0] == 0 else "salmon"
        ax.axvspan(
            dates[t0], dates[min(t, T - 1)],
            color=color, alpha=0.22, linewidth=0, zorder=0,
        )
        t0 = t

ax.plot(dates, spy_px, color="navy", lw=1.15, zorder=2, label="SPY")
ax.set_ylabel("SPY")
ax.set_title("SPY price with regime background (green = Normal, red = Stress)")
ax.legend(loc="upper left", framealpha=0.9)

# Optional: light grid only on price panel
ax.grid(True, alpha=0.25)

fig.autofmt_xdate()
plt.tight_layout()
plt.savefig("regime_probabilities_weekly_K2_asymm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved -> regime_probabilities_weekly_K2_asymm.png")


## 7. Expanding-Window Out-of-Sample Evaluation

In [ ]:
# Trading probabilities — EMA smoother (implementation layer only)
# Statistical inference still uses xi_filtered / xi_smoothed from Cell 5.
# Position engine uses pi_trade to reduce week-to-week flutter.

LAM = 0.85 # 0.85–0.95; higher = smoother / stickier

pi_trade = np.zeros((T, K))
pi_trade[0] = xi_filtered[0].copy()
for t in range(1, T):
    pi_trade[t] = LAM * pi_trade[t - 1] + (1.0 - LAM) * xi_filtered[t]
    pi_trade[t] /= pi_trade[t].sum()

def mean_abs_diff(p):
    return float(np.mean(np.abs(np.diff(p[:, 0]))))

print(f"Mean |ΔP(Normal)| filtered : {mean_abs_diff(xi_filtered):.4f}")
print(f"Mean |ΔP(Normal)| trade EMA: {mean_abs_diff(pi_trade):.4f}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(weekly.index, xi_smoothed[:, 0], color="lightgreen", alpha=0.7, lw=0.8,
        label="Smoothed P(Normal)")
ax.plot(weekly.index, pi_trade[:, 0], color="darkgreen", lw=1.2,
        label=f"Trade EMA λ={LAM}")
ax.set_ylim(0, 1.05)
ax.legend()


## 8. Additional Analysis

In [ ]:
# Cell 8 (revised): Risk-minimization allocation
# Design for a risk-minimization paper:
#   - Non-stress: fully invested in SPY (weight = 1)
#   - Stress: defensive weight W_STRESS in [0, 1)
# Statistical filter unchanged; this cell is the implementation layer only.
# A smoother P(Normal)->weight map can replace the discrete rule later.

from scipy.stats import norm as _norm

# ---------------------------------------------------------------------------
# Mixture Expected Shortfall under Gaussian regime components
# Portfolio return approx: R = w * r_equity  (cash leg treated as ~0 for ES)
# R | regime k  ~  N(w * mu[k],  w^2 * sigma2[k])
# ---------------------------------------------------------------------------
def mixture_es_gaussian(w, probs, mu_vec, s2_vec, alpha=0.05, n_grid=800):
    """
    Approximate portfolio Expected Shortfall (CVaR) for a discrete Gaussian mixture.

    Parameters
    ----------
    w : float
        Equity weight in [0, 1].
    probs : array-like, shape (K,)
        Regime probabilities (sum to 1).
    mu_vec : array-like, shape (K,)
        Regime-conditional means of equity return.
    s2_vec : array-like, shape (K,)
        Regime-conditional variances of equity return.
    alpha : float
        Tail probability (e.g. 0.05 -> 5% ES).
    n_grid : int
        Grid size for numerical mixture density.

    Returns
    -------
    float
        Approximate ES (typically negative). More negative = worse tail.
    """
    w = float(w)
    probs = np.asarray(probs, dtype=float)
    mu_vec = np.asarray(mu_vec, dtype=float)
    s2_vec = np.asarray(s2_vec, dtype=float)
    K_ = len(probs)

    port_mu = w * mu_vec
    port_std = np.abs(w) * np.sqrt(np.maximum(s2_vec, 1e-16))

    # Adaptive grid covering the left tail across all components
    left = float(np.min(port_mu - 6.0 * port_std))
    right = float(np.max(port_mu + 2.0 * port_std))
    if not np.isfinite(left) or not np.isfinite(right) or right <= left:
        left, right = -0.30, 0.10
    grid = np.linspace(left, right, n_grid)
    dens = np.zeros(n_grid, dtype=float)
    for k in range(K_):
        s = max(port_std[k], 1e-12)
        dens += probs[k] * _norm.pdf(grid, loc=port_mu[k], scale=s)
    mass = dens.sum()
    if mass < 1e-300:
        return float(port_mu @ probs)  # degenerate fallback
    dens /= mass
    cdf = np.cumsum(dens)
    idx = int(np.searchsorted(cdf, alpha))
    idx = min(max(idx, 1), n_grid - 1)
    # Tail expectation
    tail = dens[: idx + 1]
    if tail.sum() < 1e-300:
        return float(grid[idx])
    es = float(np.average(grid[: idx + 1], weights=tail))
    return es


USE_TRADE_PI = True          # True -> pi_trade; False -> xi_filtered
MODE = "hard"                # "hard" or "soft"

TAU_STRESS = 0.50            # stress threshold on P(Stress)
W_STRESS   = 0.00            # 0.00 = cash/BIL; try 0.25 / 0.50 as robustness
W_NORMAL   = 1.00            # full SPY when not in stress
CVAR_ALPHA = 0.05
CVAR_LIMIT = -0.15           # optional extra safety; set very low to disable
USE_CVAR   = False           # hard regime rule is already defensive

pi_src = pi_trade if USE_TRADE_PI else xi_filtered

w_target = np.zeros(T)
es_path  = np.zeros(T)
p_normal = np.zeros(T)
p_stress = np.zeros(T)

for t in range(T):
    xi_t = pi_src[t]
    p_n = float(xi_t[0])
    p_s = float(xi_t[1])
    p_normal[t] = p_n
    p_stress[t] = p_s

    if MODE == "hard":
        # Binary risk rule: full risk-on unless stress probability exceeds threshold
        w = W_STRESS if (p_s > TAU_STRESS) else W_NORMAL
    else:
        # Soft rule: full SPY until TAU_STRESS, then interpolate down to W_STRESS
        if p_s <= TAU_STRESS:
            w = W_NORMAL
        else:
            # p_s in (TAU_STRESS, 1] -> w in (W_NORMAL, W_STRESS]
            alpha = (p_s - TAU_STRESS) / max(1e-12, 1.0 - TAU_STRESS)
            w = W_NORMAL + alpha * (W_STRESS - W_NORMAL)
        w = float(np.clip(w, min(W_STRESS, W_NORMAL), max(W_STRESS, W_NORMAL)))

    if USE_CVAR:
        es = mixture_es_gaussian(w, xi_t, mu, sigma2_path[t], CVAR_ALPHA)
        if es < CVAR_LIMIT:
            lo, hi, best = min(W_STRESS, W_NORMAL), w, min(W_STRESS, W_NORMAL)
            for _ in range(25):
                mid = 0.5 * (lo + hi)
                if mixture_es_gaussian(mid, xi_t, mu, sigma2_path[t], CVAR_ALPHA) >= CVAR_LIMIT:
                    best, lo = mid, mid
                else:
                    hi = mid
            w = best
        es_path[t] = mixture_es_gaussian(w, xi_t, mu, sigma2_path[t], CVAR_ALPHA)
    else:
        es_path[t] = mixture_es_gaussian(w, xi_t, mu, sigma2_path[t], CVAR_ALPHA)

    w_target[t] = float(w)

weekly = weekly.copy()
weekly["w_target"] = w_target
weekly["ES"] = es_path
weekly["p_normal"] = p_normal
weekly["p_stress"] = p_stress

print(f"MODE={MODE}  TAU_STRESS={TAU_STRESS}  W_STRESS={W_STRESS}  W_NORMAL={W_NORMAL}")
print(weekly[["w_target", "p_normal", "p_stress", "ES"]].describe().round(4))
print(f"Mean weight = {w_target.mean():.3f}")
print(f"Frac at full SPY (w=1): {(np.abs(w_target - 1.0) < 1e-12).mean():.1%}")
print(f"Frac at W_STRESS:       {(np.abs(w_target - W_STRESS) < 1e-12).mean():.1%}")


## 9. Additional Analysis

In [ ]:
# Asymmetric hysteresis + minimum holding (risk-min rule)
# Enter defensive mode more easily than returning to 100% SPY
# (risk-management asymmetry: cut risk fast, re-risk slow).

BAND_TO_STRESS = 0.08    # smaller gap enough to cut toward W_STRESS
BAND_TO_FULL   = 0.15    # larger gap needed to return to 100% SPY
MIN_HOLD_WEEKS = 4

w_exec = np.zeros(T)
w_exec[0] = w_target[0]
last_change = 0

for t in range(1, T):
    target  = w_target[t]
    current = w_exec[t - 1]
    held    = t - last_change

    if held < MIN_HOLD_WEEKS:
        w_exec[t] = current
        continue

    # Moving down (increasing risk control): easier
    if target < current - BAND_TO_STRESS:
        w_exec[t] = target
        last_change = t
    # Moving up (back toward 100% SPY): harder
    elif target > current + BAND_TO_FULL:
        w_exec[t] = target
        last_change = t
    else:
        w_exec[t] = current

weekly["w_exec"] = w_exec
n_trades = int(np.sum(np.abs(np.diff(w_exec)) > 1e-8))
print(f"Executed trades: {n_trades} over {T} weeks")
print(f"Average holding between changes: {T / max(n_trades, 1):.1f} weeks")
print(f"Exec mean weight = {w_exec.mean():.3f}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(weekly.index, weekly["w_target"], color="gray", alpha=0.55, lw=0.9, label="Target")
ax.plot(weekly.index, weekly["w_exec"], color="darkred", lw=1.2, label="Executed")
ax.axhline(1.0, color="green", ls="--", lw=0.8, alpha=0.7, label="Full SPY")
ax.axhline(W_STRESS, color="salmon", ls="--", lw=0.8, alpha=0.8, label=f"W_STRESS={W_STRESS}")
ax.set_ylabel("Weight on SPY")
ax.set_ylim(-0.05, 1.15)
ax.set_title("Risk-min rule: 100% SPY in Normal, defensive weight in Stress")
ax.legend(loc="lower left", ncol=2)
plt.tight_layout()
plt.savefig("weights_riskmin_weekly_K2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved -> weights_riskmin_weekly_K2.png")


## 10. Additional Analysis

In [ ]:
# Strategy returns with transaction costs + simple tax drag
# Gross: w_{t-1} * SPY_return_t + (1 - w_{t-1}) * BIL_return_t
# Cost charged on |Δw| when weight changes.
# Tax: approximate ST rate on gains when holding < TAX_LT_WEEKS.

COST_BPS       = 5.0          # one-way cost in basis points of notional traded
TAX_ST_RATE    = 0.37         # short-term capital gains (illustrative US)
TAX_LT_RATE    = 0.20         # long-term
TAX_LT_WEEKS   = 52           # holding weeks to qualify as LT (approx)

spy_ret = weekly["SPY_return"].values.astype(np.float64)
bil_ret = weekly["BIL_return"].values.astype(np.float64)

# Align: weight decided at t-1 applies to return at t
w_lag = np.roll(w_exec, 1)
w_lag[0] = w_exec[0]

gross_ret = w_lag * spy_ret + (1.0 - w_lag) * bil_ret

# Transaction costs on weight changes
dw = np.abs(np.diff(w_exec, prepend=w_exec[0]))
cost = dw * (COST_BPS / 10000.0)
net_ret = gross_ret - cost

# Simple tax drag: when a trade reduces exposure after a gain, apply tax rate
# based on holding length since last change (stylised; not lot-level).
tax_drag = np.zeros(T)
last_ch  = 0
for t in range(1, T):
    if abs(w_exec[t] - w_exec[t - 1]) > 1e-8:
        held = t - last_ch
        # Approximate taxable base: positive contribution from equity sleeve
        equity_pnl = w_exec[t - 1] * spy_ret[t]
        if equity_pnl > 0 and w_exec[t] < w_exec[t - 1]:
            rate = TAX_LT_RATE if held >= TAX_LT_WEEKS else TAX_ST_RATE
            tax_drag[t] = rate * equity_pnl * (w_exec[t - 1] - w_exec[t])
        last_ch = t

net_ret_after_tax = net_ret - tax_drag
bh = spy_ret.copy()

def perf_stats(r, ann_factor=52.0):
    r = np.asarray(r, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 2:
        return dict(AnnReturn=np.nan, AnnVol=np.nan, Sharpe=np.nan,
                    Sortino=np.nan, MaxDD=np.nan, N=len(r))
    mu_a = r.mean() * ann_factor
    vol_a = r.std(ddof=1) * np.sqrt(ann_factor)
    downside = r[r < 0]
    dvol = downside.std(ddof=1) * np.sqrt(ann_factor) if len(downside) > 1 else np.nan
    sharpe = mu_a / vol_a if vol_a > 1e-12 else np.nan
    sortino = mu_a / dvol if dvol and dvol > 1e-12 else np.nan
    wealth = np.cumprod(1.0 + r)
    peak = np.maximum.accumulate(wealth)
    maxdd = float((wealth / peak - 1.0).min())
    return dict(AnnReturn=mu_a, AnnVol=vol_a, Sharpe=sharpe,
                Sortino=sortino, MaxDD=maxdd, N=len(r))

rows = {
    "BuyHold":           perf_stats(bh),
    "Strategy_gross":    perf_stats(gross_ret),
    "Strategy_net":      perf_stats(net_ret),
    "Strategy_net_tax":  perf_stats(net_ret_after_tax),
}
perf_df = pd.DataFrame(rows).T
print(perf_df.round(4))
perf_df.to_csv("performance_fullsample_weekly.csv")
print("Saved → performance_fullsample_weekly.csv")

turnover_ann = float(np.sum(np.abs(np.diff(w_exec))) * (52.0 / T))
print(f"Mean annual turnover (sum |Δw| * 52/T): {turnover_ann:.2f}")


## 11. Additional Analysis

In [ ]:
# Equity curves and drawdowns (full-sample illustration)
# NOTE: Full-sample weights use the full-sample filter — illustrative only.
# Strict no-look-ahead OOS is in the next cell.

def wealth_and_dd(r):
    w = np.cumprod(1.0 + np.asarray(r))
    peak = np.maximum.accumulate(w)
    dd = w / peak - 1.0
    return w, dd

w_bh, dd_bh = wealth_and_dd(bh)
w_st, dd_st = wealth_and_dd(net_ret_after_tax)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(weekly.index, w_bh, color="gray", lw=1.2, label="Buy & Hold")
axes[0].plot(weekly.index, w_st, color="darkred", lw=1.2, label="Strategy (net+tax)")
axes[0].set_ylabel("Growth of $1")
axes[0].legend()
axes[0].set_title("Full-sample equity curves (illustrative; see OOS cell for strict evaluation)")

axes[1].plot(weekly.index, dd_bh, color="gray", lw=1.0, label="Buy & Hold")
axes[1].plot(weekly.index, dd_st, color="darkred", lw=1.0, label="Strategy")
axes[1].set_ylabel("Drawdown")
axes[1].legend()
plt.tight_layout()
plt.savefig("equity_drawdown_fullsample.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → equity_drawdown_fullsample.png")


## 12. Additional Analysis

In [ ]:
# Out-of-Sample skeleton (strict causality on inputs)
# Practical path: parameters locked from full-sample MLE;
# filtered probabilities and weights use only information available at t
# (lagged inversion + past residuals in GJR).
# For the paper, set RUN_FULL_ROLLING = True and re-estimate on expanding
# windows (computationally heavier).

TRAIN_END_RATIO = 0.50
RUN_FULL_ROLLING = False   # True = re-estimate every REEST_EVERY weeks

n_train = int(T * TRAIN_END_RATIO)
print(f"Initial train window: weeks [0, {n_train})  |  OOS weeks [{n_train}, {T})")

idx = np.arange(T) >= n_train
stats_oos    = perf_stats(net_ret_after_tax[idx])
stats_bh_oos = perf_stats(bh[idx])
print("\nOOS period performance (parameters locked from full-sample MLE – preliminary):")
print(pd.DataFrame({"Strategy_OOS": stats_oos, "BuyHold_OOS": stats_bh_oos}).T.round(4))

print("""
NOTE
----
This preliminary OOS uses full-sample MLE parameters (filter remains causal in inputs).
For the paper:
  - Re-estimate on expanding windows (RUN_FULL_ROLLING=True), or
  - Rolling windows of 300–500 weeks,
  then rebuild weights only with time-t information.
Ablation stack to implement next:
  1. Buy & Hold
  2. Constant vol targeting
  3. Conditional vol targeting
  4. Constant-transition MS-GARCH
  5. Parsimonious TVTP without hysteresis
  6. Full master strategy (TVTP + hysteresis + CVaR)
""")


## 13. Additional Analysis

In [ ]:
# Monte Carlo skeleton (finite-sample classification, K=2)
# Quick upper bound: filter with true parameters (no re-estimation).
# For the paper, add full re-estimation MC (bias / RMSE / F1).

N_MC   = 20
T_LIST = [200, 400, 800]

def simulate_path(T_sim, seed):
    rng = np.random.default_rng(seed)
    inv = np.zeros(T_sim)
    inv[0] = 1.0
    for t in range(1, T_sim):
        inv[t] = 0.90 * inv[t-1] + 0.10 + 0.04 * rng.normal()
        inv[t] = np.clip(inv[t], 0.65, 1.80)

    ret_s  = np.zeros(T_sim)
    states = np.zeros(T_sim, dtype=int)
    s2     = np.full(K, 1e-4)
    states[0] = 0
    ret_s[0]  = mu[0] + np.sqrt(s2[0]) * rng.normal()

    for t in range(1, T_sim):
        P = transition_matrix(inv[t-1], trans_intercept, slope_vec)
        states[t] = rng.choice(K, p=P[states[t-1]])
        eps = ret_s[t-1] - mu[states[t-1]]
        Ineg = 1.0 if eps < 0 else 0.0
        k = states[t]
        s2[k] = omega[k] + (alpha[k] + gamma[k] * Ineg) * eps**2 + beta[k] * s2[k]
        s2[k] = max(s2[k], 1e-12)
        ret_s[t] = mu[k] + np.sqrt(s2[k]) * rng.normal()
    return ret_s, states, inv


def filtered_states_given(ret_s, inv_lag, mu_e, omega_e, alpha_e, gamma_e, beta_e,
                          t_int, s_vec):
    T_sim = len(ret_s)
    xi = np.ones(K) / K
    s2 = np.full(K, np.var(ret_s[:min(30, T_sim)]))
    hat = np.zeros(T_sim, dtype=int)
    for t in range(T_sim):
        P = transition_matrix(inv_lag[t], t_int, s_vec)
        xp = P.T @ xi
        dens = np.zeros(K)
        for k in range(K):
            dens[k] = np.exp(-0.5 * np.log(2 * np.pi * s2[k])
                             - 0.5 * (ret_s[t] - mu_e[k])**2 / s2[k])
        dens = np.maximum(dens, 1e-300)
        joint = xp * dens
        xi = joint / joint.sum()
        hat[t] = np.argmax(xi)
        for k in range(K):
            eps = ret_s[t] - mu_e[k]
            Ineg = 1.0 if eps < 0 else 0.0
            s2[k] = omega_e[k] + (alpha_e[k] + gamma_e[k] * Ineg) * eps**2 + beta_e[k] * s2[k]
            s2[k] = max(s2[k], 1e-12)
    return hat


try:
    from sklearn.metrics import f1_score
except ImportError:
    def f1_score(y, yhat, average="macro", zero_division=0):
        scores = []
        for c in range(K):
            tp = np.sum((y == c) & (yhat == c))
            fp = np.sum((y != c) & (yhat == c))
            fn = np.sum((y == c) & (yhat != c))
            prec = tp / (tp + fp) if (tp + fp) else 0.0
            rec  = tp / (tp + fn) if (tp + fn) else 0.0
            scores.append(0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec))
        return float(np.mean(scores))

print("Quick MC (K=2): filter with true parameters (upper-bound Macro-F1)\n")
for T_sim in T_LIST:
    f1s = []
    for mc in range(N_MC):
        ret_s, st, inv = simulate_path(T_sim, seed=RANDOM_SEED + mc * 17)
        inv_lag = np.roll(inv, 1)
        inv_lag[0] = inv[0]
        hat = filtered_states_given(
            ret_s, inv_lag, mu, omega, alpha, gamma, beta,
            trans_intercept, slope_vec
        )
        f1s.append(f1_score(st, hat, average="macro", zero_division=0))
    print(f"T={T_sim:4d}  Mean Macro-F1 = {np.mean(f1s):.3f}  Std = {np.std(f1s):.3f}")

print("""
Next for the paper
------------------
1. Full re-estimation MC (Bias, RMSE, F1)
2. Expanding-window OOS ablation + Ledoit-Wolf
3. Sensitivity: MIN_WEIGHT, BAND_*, MIN_HOLD_WEEKS, TARGET_STAY, PEN_WEIGHT
""")


## 14. Additional Analysis

In [ ]:
# Expanding-window OOS with parameter re-estimation
# Strict no-look-ahead:
#   - At each re-estimation date t_reest, MLE uses only data [0, t_reest)
#   - From t_reest until the next re-estimation, filtered probs and weights
#     use only the locked parameters + causal inputs (lagged inversion, past residuals)
# Soft persistence penalty is the same family as the full-sample objective.
#
# Position rule (aligned with Cell 8 risk-minimization):
#   - Non-stress: full SPY (W_NORMAL_OOS = 1)
#   - Stress: defensive weight W_STRESS_OOS in [0, 1)
#   MODE_OOS = "hard" | "soft"  (soft interpolates after the stress threshold;
#   a fully continuous P(Normal)->weight map can replace this later)
#
# Runtime note:
#   Full multi-start every step is slow. Defaults use warm-start from the
#   previous estimate + a small number of extra random starts.
#   For the final paper run, raise N_STARTS_OOS and/or lower REEST_EVERY.

import time
from copy import deepcopy

# ---------------------------------------------------------------------------
# Expanding OOS configuration
# ---------------------------------------------------------------------------
TRAIN_MIN_WEEKS = 300       # first OOS weight appears at this week index
REEST_EVERY     = 26        # re-estimate every N weeks
N_STARTS_OOS    = 4         # random starts per re-estimation (+ optional warm)
MAX_ITER_OOS    = 250
FTOL_OOS        = 1e-8

# Position rule (risk-minimization; matches Cell 8)
MODE_OOS        = "hard"    # "hard" or "soft"
TAU_STRESS_OOS  = 0.50      # P(Stress) threshold
W_STRESS_OOS    = 0.00      # defensive equity weight in Stress (0 = full BIL)
W_NORMAL_OOS    = 1.00      # full SPY when not in Stress

# Trading EMA + asymmetric hysteresis
LAM_OOS         = 0.85
BAND_UP_OOS     = 0.12
BAND_DOWN_OOS   = 0.08
MIN_HOLD_OOS    = 4

# Optional CVaR guardrail (usually off when hard regime rule is already defensive)
USE_CVAR_OOS    = False
CVAR_ALPHA_OOS  = 0.05
CVAR_LIMIT_OOS  = -0.15

# Costs and stylised tax
COST_BPS_OOS     = 5.0
TAX_ST_OOS       = 0.37
TAX_LT_OOS       = 0.20
TAX_LT_WEEKS_OOS = 52


# ---------------------------------------------------------------------------
# Segment NLL (soft persistence; no Stress-mass term to keep OOS light)
# ---------------------------------------------------------------------------
def nll_expanding(theta, ret_seg, inv_seg):
    """NLL + soft persistence penalty on a data segment (no globals except K)."""
    mu_u, omega_u, alpha_u, gamma_u, beta_u, t_int, s_vec = unpack_parameters(theta)
    if not (omega_u[0] < omega_u[1]):
        return 1e12 + 1e6 * max(0.0, omega_u[0] - omega_u[1])
    P_all = build_P_all(inv_seg, t_int, s_vec)
    nll = float(nll_numba(ret_seg, mu_u, omega_u, alpha_u, gamma_u, beta_u, P_all))
    stay0 = float(P_all[:, 0, 0].mean())
    stay1 = float(P_all[:, 1, 1].mean())
    TARGET_STAY0, TARGET_STAY1, PEN_WEIGHT = 0.90, 0.85, 200.0
    pen = PEN_WEIGHT * (
        max(0.0, TARGET_STAY0 - stay0) ** 2 +
        max(0.0, TARGET_STAY1 - stay1) ** 2
    )
    return nll + pen


def estimate_on_segment(ret_seg, inv_seg, warm_theta=None, n_starts=N_STARTS_OOS):
    """Multi-start MLE on [ret_seg, inv_seg]; optional warm start."""
    best_local_nll = np.inf
    best_local_theta = None
    starts = []

    if warm_theta is not None:
        starts.append(np.asarray(warm_theta, dtype=float).copy())

    rng = np.random.default_rng(RANDOM_SEED + len(ret_seg))
    for s in range(n_starts):
        mu_init = np.array([0.0025, -0.0030]) + rng.normal(0.0, 0.0010, K)
        omega_init = np.log(np.array([1.5e-5, 8e-5]) * (0.5 + rng.random(K)))
        alpha_init = np.log(np.clip(np.array([0.05, 0.10]) * (0.5 + rng.random(K)), 1e-4, 0.45))
        gamma_init = np.array([0.10, 0.25]) + rng.normal(0.0, 0.05, K)
        beta_raw = np.clip(np.array([0.80, 0.70]) + rng.normal(0.0, 0.05, K), 0.50, 0.96)
        beta_init = np.log(beta_raw / (1.0 - beta_raw))
        inter_init = rng.normal(0.0, 0.30, (K, K))
        np.fill_diagonal(inter_init, 2.0 + 0.8 * rng.random(K))
        raw_enter = rng.normal(-0.4, 0.4)
        raw_exit  = rng.normal(0.0, 0.4)
        starts.append(np.concatenate([
            mu_init, omega_init, alpha_init, gamma_init, beta_init,
            inter_init.ravel(), np.array([raw_enter, raw_exit]),
        ]))

    for theta0 in starts:
        try:
            res = minimize(
                nll_expanding, theta0, args=(ret_seg, inv_seg),
                method="L-BFGS-B",
                options={"maxiter": MAX_ITER_OOS, "ftol": FTOL_OOS, "disp": False},
            )
            if np.isfinite(res.fun) and res.fun < best_local_nll:
                best_local_nll = float(res.fun)
                best_local_theta = res.x.copy()
        except Exception:
            continue

    return best_local_theta, best_local_nll


def filter_segment(ret_seg, inv_seg, theta):
    """Causal Hamilton filter on a segment; returns xi_filtered, sigma2_path."""
    mu_e, omega_e, alpha_e, gamma_e, beta_e, t_int, s_vec = unpack_parameters(theta)
    n = len(ret_seg)
    xi_f = np.zeros((n, K))
    s2_path = np.zeros((n, K))
    xi = np.ones(K) / K
    s2 = np.full(K, np.var(ret_seg[:min(50, n)]))
    for t in range(n):
        P = transition_matrix(inv_seg[t], t_int, s_vec)
        xi_pred = P.T @ xi
        if t > 0:
            eps = ret_seg[t - 1] - mu_e
            Ineg = (eps < 0).astype(float)
        else:
            eps = np.zeros(K)
            Ineg = np.zeros(K)
        s2n = omega_e + (alpha_e + gamma_e * Ineg) * (eps ** 2) + beta_e * s2
        s2n = np.maximum(s2n, 1e-12)
        s2_path[t] = s2n
        resid = ret_seg[t] - mu_e
        dens = np.exp(-0.5 * np.log(2 * np.pi * s2n) - 0.5 * (resid ** 2) / s2n)
        dens = np.maximum(dens, 1e-300)
        joint = xi_pred * dens
        s = joint.sum()
        xi = joint / s if s > 1e-300 else np.ones(K) / K
        s2 = s2n
        xi_f[t] = xi
    return xi_f, s2_path, (mu_e, omega_e, alpha_e, gamma_e, beta_e, t_int, s_vec)


def target_weight_from_pi(pi_t, mu_e, s2_t):
    """
    Risk-minimization map (Cell 8 style).
    hard: P(Stress) > TAU -> W_STRESS, else W_NORMAL
    soft: full SPY until TAU, then linear interpolation down to W_STRESS
    Optional CVaR guardrail can scale weight down further.
    """
    p_s = float(pi_t[1])

    if MODE_OOS == "hard":
        w = W_STRESS_OOS if (p_s > TAU_STRESS_OOS) else W_NORMAL_OOS
    else:
        # soft transition after the stress threshold
        if p_s <= TAU_STRESS_OOS:
            w = W_NORMAL_OOS
        else:
            alpha = (p_s - TAU_STRESS_OOS) / max(1e-12, 1.0 - TAU_STRESS_OOS)
            w = W_NORMAL_OOS + alpha * (W_STRESS_OOS - W_NORMAL_OOS)
        lo_w = min(W_STRESS_OOS, W_NORMAL_OOS)
        hi_w = max(W_STRESS_OOS, W_NORMAL_OOS)
        w = float(np.clip(w, lo_w, hi_w))

    if USE_CVAR_OOS:
        es = mixture_es_gaussian(w, pi_t, mu_e, s2_t, CVAR_ALPHA_OOS)
        if es < CVAR_LIMIT_OOS:
            lo, hi, best = min(W_STRESS_OOS, W_NORMAL_OOS), w, min(W_STRESS_OOS, W_NORMAL_OOS)
            for _ in range(25):
                mid = 0.5 * (lo + hi)
                if mixture_es_gaussian(mid, pi_t, mu_e, s2_t, CVAR_ALPHA_OOS) >= CVAR_LIMIT_OOS:
                    best, lo = mid, mid
                else:
                    hi = mid
            w = best

    return float(np.clip(w, min(W_STRESS_OOS, W_NORMAL_OOS), max(W_STRESS_OOS, W_NORMAL_OOS)))


# -------------------- expanding schedule --------------------
reest_dates = list(range(TRAIN_MIN_WEEKS, T, REEST_EVERY))
if len(reest_dates) == 0:
    raise ValueError("TRAIN_MIN_WEEKS >= T; lower TRAIN_MIN_WEEKS or check sample length.")

print(f"Expanding OOS setup")
print(f"  T = {T}  |  first OOS weights from week {TRAIN_MIN_WEEKS}")
print(f"  Re-estimate every {REEST_EVERY} weeks  |  n re-estimations = {len(reest_dates)}")
print(f"  Starts per re-est = {N_STARTS_OOS} + warm-start")
print(f"  Position: MODE={MODE_OOS}  TAU={TAU_STRESS_OOS}  W_STRESS={W_STRESS_OOS}  W_NORMAL={W_NORMAL_OOS}")
print("=" * 72)

w_oos = np.full(T, np.nan)
w_oos_exec = np.full(T, np.nan)
theta_path = [None] * T
nll_path = np.full(len(reest_dates), np.nan)

warm = best_theta.copy() if "best_theta" in dir() and best_theta is not None else None
t0_all = time.time()

for i, t_reest in enumerate(reest_dates):
    # ----- estimate on [0, t_reest) -----
    ret_seg = returns[:t_reest]
    inv_seg = inversion_lagged[:t_reest]
    t_est0 = time.time()
    theta_hat, nll_hat = estimate_on_segment(
        ret_seg, inv_seg, warm_theta=warm, n_starts=N_STARTS_OOS
    )
    dt_est = time.time() - t_est0
    if theta_hat is None:
        print(f"[{i+1}/{len(reest_dates)}] t={t_reest}: estimation failed — keep previous theta")
        theta_hat = warm
        nll_hat = np.nan
    else:
        warm = theta_hat.copy()
    nll_path[i] = nll_hat

    # apply this theta from t_reest until next reest (or T)
    t_end = reest_dates[i + 1] if (i + 1) < len(reest_dates) else T

    # filter on [0, t_end) with locked theta; use weights only on [t_reest, t_end)
    xi_f, s2_path, unpacked = filter_segment(
        returns[:t_end], inversion_lagged[:t_end], theta_hat
    )
    mu_e = unpacked[0]

    # trading EMA only within the application window (causal recursion)
    if t_reest > 0:
        pi = xi_f[t_reest - 1].copy()
    else:
        pi = np.ones(K) / K

    for t in range(t_reest, t_end):
        pi = LAM_OOS * pi + (1.0 - LAM_OOS) * xi_f[t]
        pi = pi / pi.sum()
        w_oos[t] = target_weight_from_pi(pi, mu_e, s2_path[t])
        theta_path[t] = theta_hat

    print(
        f"[{i+1:02d}/{len(reest_dates)}] reest @ week {t_reest:4d} → apply through {t_end-1:4d}  "
        f"| NLL+pen={nll_hat:10.2f}  ({dt_est:.1f}s)"
    )

print("=" * 72)
print(f"All re-estimations done in {(time.time() - t0_all)/60:.1f} min")

# -------------------- hysteresis on OOS target weights --------------------
first = TRAIN_MIN_WEEKS
w_oos_exec[first] = w_oos[first]
last_change = first
for t in range(first + 1, T):
    if not np.isfinite(w_oos[t]):
        w_oos_exec[t] = w_oos_exec[t - 1]
        continue
    target = w_oos[t]
    current = w_oos_exec[t - 1]
    held = t - last_change
    if held < MIN_HOLD_OOS:
        w_oos_exec[t] = current
        continue
    if target > current + BAND_UP_OOS:
        w_oos_exec[t] = target
        last_change = t
    elif target < current - BAND_DOWN_OOS:
        w_oos_exec[t] = target
        last_change = t
    else:
        w_oos_exec[t] = current

# -------------------- OOS returns (cost + stylised tax) --------------------
w_lag = np.roll(w_oos_exec, 1)
w_lag[first] = w_oos_exec[first]

gross = w_lag * spy_ret + (1.0 - w_lag) * bil_ret
dw = np.abs(np.diff(w_oos_exec, prepend=w_oos_exec[0]))
cost = dw * (COST_BPS_OOS / 10000.0)
net = gross - cost

tax_drag = np.zeros(T)
last_ch = first
for t in range(first + 1, T):
    if abs(w_oos_exec[t] - w_oos_exec[t - 1]) > 1e-8:
        held = t - last_ch
        equity_pnl = w_oos_exec[t - 1] * spy_ret[t]
        if equity_pnl > 0 and w_oos_exec[t] < w_oos_exec[t - 1]:
            rate = TAX_LT_OOS if held >= TAX_LT_WEEKS_OOS else TAX_ST_OOS
            tax_drag[t] = rate * equity_pnl * (w_oos_exec[t - 1] - w_oos_exec[t])
        last_ch = t

net_tax = net - tax_drag

# mask: only evaluate pure OOS period
oos_mask = np.arange(T) >= first
oos_mask = oos_mask & np.isfinite(net_tax) & np.isfinite(spy_ret)

stats_st = perf_stats(net_tax[oos_mask])
stats_bh = perf_stats(spy_ret[oos_mask])
stats_net = perf_stats(net[oos_mask])
stats_gr = perf_stats(gross[oos_mask])

oos_table = pd.DataFrame({
    "BuyHold_OOS": stats_bh,
    "Strategy_gross_OOS": stats_gr,
    "Strategy_net_OOS": stats_net,
    "Strategy_net_tax_OOS": stats_st,
}).T

print("\nStrict expanding-window OOS performance")
print(oos_table.round(4))
oos_table.to_csv("performance_oos_expanding.csv")
print("Saved → performance_oos_expanding.csv")

turnover_oos = float(np.nansum(np.abs(np.diff(w_oos_exec[first:]))) * (52.0 / max(T - first, 1)))
print(f"OOS annual turnover ≈ {turnover_oos:.2f}")
print(f"OOS mean weight ≈ {np.nanmean(w_oos_exec[first:]):.3f}")

# -------------------- equity / DD for OOS only --------------------
def wealth_from(r, mask):
    x = np.asarray(r[mask], dtype=float)
    w = np.cumprod(1.0 + x)
    peak = np.maximum.accumulate(w)
    return w, w / peak - 1.0

w_bh, dd_bh = wealth_from(spy_ret, oos_mask)
w_st, dd_st = wealth_from(net_tax, oos_mask)
idx_oos = weekly.index[oos_mask]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(idx_oos, w_bh, color="gray", lw=1.2, label="Buy & Hold")
axes[0].plot(idx_oos, w_st, color="darkred", lw=1.2, label="Strategy (net+tax)")
axes[0].set_ylabel("Growth of $1")
axes[0].legend()
axes[0].set_title("Expanding-window OOS equity (re-estimated parameters)")

axes[1].plot(idx_oos, dd_bh, color="gray", lw=1.0, label="Buy & Hold")
axes[1].plot(idx_oos, dd_st, color="darkred", lw=1.0, label="Strategy")
axes[1].set_ylabel("Drawdown")
axes[1].legend()
plt.tight_layout()
plt.savefig("equity_drawdown_oos_expanding.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → equity_drawdown_oos_expanding.png")
